In [10]:
import pandas as pd
import numpy as np

cmapss_path = "/kaggle/input/nasa-turbofan-engine-degradation-simulation/"

cols = ['engine_id','cycle'] + \
       [f'op_setting_{i}' for i in range(1,4)] + \
       [f'sensor_{i}' for i in range(1,22)]

train_df = pd.read_csv(cmapss_path + "train_FD001.txt", sep=r"\s+", header=None, names=cols)

In [11]:
max_cycle = train_df.groupby('engine_id')['cycle'].max().reset_index()
max_cycle.columns = ['engine_id','max_cycle']
train_df = train_df.merge(max_cycle,on='engine_id')
train_df['RUL'] = train_df['max_cycle'] - train_df['cycle']
train_df['RUL'] = train_df['RUL'].clip(upper=125)
train_df.drop(columns=['max_cycle'],inplace=True)

In [12]:
from sklearn.preprocessing import MinMaxScaler

sensor_cols=[c for c in train_df.columns if 'sensor_' in c]

scaler = MinMaxScaler()
train_df[sensor_cols] = scaler.fit_transform(train_df[sensor_cols])

In [13]:
from scipy.fft import fft
from scipy.stats import entropy

def fft_features(signal):
    fft_vals = np.abs(fft(signal))
    fft_vals = fft_vals[:len(fft_vals)//2]

    dom_freq = np.argmax(fft_vals)
    spec_energy = np.sum(fft_vals**2)
    freq_entropy = entropy(fft_vals + 1e-10)

    return dom_freq, spec_energy, freq_entropy

In [14]:
WINDOW=30

def create_windows_with_fft(df, window):
    X,y=[],[]

    for eid in df.engine_id.unique():
        d = df[df.engine_id==eid].sort_values('cycle')
        data = d[sensor_cols].values
        rul = d['RUL'].values

        for i in range(len(data)-window):
            win = data[i:i+window]
            fft_feats=[]

            for s in range(win.shape[1]):
                f1,f2,f3 = fft_features(win[:,s])
                fft_feats.extend([f1,f2,f3])

            combined = np.concatenate([win.flatten(),fft_feats])
            X.append(combined)
            y.append(rul[i+window])

    return np.array(X),np.array(y)

X,y = create_windows_with_fft(train_df,WINDOW)
print(X.shape,y.shape)

(17631, 693) (17631,)


In [15]:
import os
import pandas as pd

skab_base_path = "/kaggle/input/skoltech-anomaly-benchmark-skab/"

skab_dfs = []

for root, dirs, files in os.walk(skab_base_path):
    for file in files:
        if file.endswith(".csv"):
            full_path = os.path.join(root, file)
            df = pd.read_csv(full_path, sep=';')
            df['source_file'] = file
            skab_dfs.append(df)

skab_df = pd.concat(skab_dfs, ignore_index=True)

print("SKAB shape:", skab_df.shape)
print(skab_df.columns)

SKAB shape: (46860, 12)
Index(['datetime', 'Accelerometer1RMS', 'Accelerometer2RMS', 'Current',
       'Pressure', 'Temperature', 'Thermocouple', 'Voltage',
       'Volume Flow RateRMS', 'anomaly', 'changepoint', 'source_file'],
      dtype='object')


In [16]:
possible_time_cols = ['datetime', 'time', 't', 'timestamp']

time_col = None
for col in possible_time_cols:
    if col in skab_df.columns:
        time_col = col
        break

print("Detected time column:", time_col)

Detected time column: datetime


In [17]:
skab_df = skab_df.sort_values(by=time_col)

skab_df.fillna(method='ffill', inplace=True)
skab_df.fillna(method='bfill', inplace=True)

print("Missing after cleaning:", skab_df.isna().sum().sum())

Missing after cleaning: 0


/tmp/ipykernel_55/3047744923.py:3: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  skab_df.fillna(method='ffill', inplace=True)
/tmp/ipykernel_55/3047744923.py:4: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  skab_df.fillna(method='bfill', inplace=True)


In [19]:
import numpy as np

np.save("/kaggle/working/X_train_fft.npy", X)
np.save("/kaggle/working/y_train_fft.npy", y)

print("FFT feature files saved!")

FFT feature files saved!
